# Multiply Venn Diagram

In the blend chapter this notebook generates the venn diagram for the multiply blend mode. Cyan, magenta, and yellow circles are multiplied together over a white background, mimicking ink on paper. Compare it with the additive RGB venn diagram in the color chapter.


In [ ]:
import math
from pathlib import Path

import brushcue

CIRCLE_DIAMETER = 1000
DISTANCE_BETWEEN_CIRCLES = 500
FOREGROUND_RADIUS = CIRCLE_DIAMETER / 2
TRIANGLE_HEIGHT = DISTANCE_BETWEEN_CIRCLES * math.sqrt(3) / 2
VENN_CENTER = brushcue.Point2f.from_components(
    DISTANCE_BETWEEN_CIRCLES / 2,
    TRIANGLE_HEIGHT / 3,
)
BACKGROUND_RADIUS = FOREGROUND_RADIUS + DISTANCE_BETWEEN_CIRCLES / math.sqrt(3)

# Relative to this notebook's directory (py/brushcue/examples/book/blend).
OUTPUT_FILE = Path("../../../../../writing/graphics/chapters/blend/assets/multiply-venn-diagram.png")

# Multiply is not linear across color spaces the way add is, so every layer is
# converted into an sRGB working space before blending.
WORKING_COLOR_PROFILE = brushcue.ColorProfile.srgb()

context = brushcue.Context()

def make_circle(center: brushcue.Point2f, diameter: float, rgba: brushcue.RGBAColor) -> brushcue.Composition:
    instances = brushcue.Transform2.to_list(brushcue.Transform2.identity())
    render_style = brushcue.RenderStyle.fill_only(
        brushcue.Fill.solid(
            brushcue.ProfiledColor.from_rgba_srgb(
                rgba
            )
        )
    )
    painter = brushcue.Painter.new()
    painter = painter.add_ellipse_with_render_style(
        center,
        brushcue.Vector2f.from_components(diameter, diameter),
        0,
        render_style,
        instances
    )
    return brushcue.Composition.painter(painter).color_convert(WORKING_COLOR_PROFILE)

cyan_circle = make_circle(brushcue.Point2f.from_components(0, 0), CIRCLE_DIAMETER, brushcue.RGBAColor.from_components(0, 1, 1, 1))
magenta_circle = make_circle(brushcue.Point2f.from_components(DISTANCE_BETWEEN_CIRCLES, 0), CIRCLE_DIAMETER, brushcue.RGBAColor.from_components(1, 0, 1, 1))
yellow_circle = make_circle(brushcue.Point2f.from_components(DISTANCE_BETWEEN_CIRCLES / 2, TRIANGLE_HEIGHT), CIRCLE_DIAMETER, brushcue.RGBAColor.from_components(1, 1, 0, 1))

cyan_and_magenta = brushcue.Composition.blend_multiply(cyan_circle, magenta_circle)
cyan_and_magenta_and_yellow = brushcue.Composition.blend_multiply(cyan_and_magenta, yellow_circle)

# White is the identity for multiply, like a blank sheet of paper under the ink.
background_circle = make_circle(VENN_CENTER, BACKGROUND_RADIUS * 2, brushcue.RGBAColor.from_components(1, 1, 1, 1))
result = brushcue.Composition.blend_multiply(background_circle, cyan_and_magenta_and_yellow)

result = result.color_convert(brushcue.ColorProfile.srgb())

image_bytes = result.execute(context).to_image_bytes(context)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE, "wb") as file:
    file.write(bytes(image_bytes))
